https://python.langchain.com/v0.2/docs/tutorials/rag/
https://python.langchain.com/v0.2/docs/integrations/chat_loaders/langsmith_llm_runs/

https://docs.smith.langchain.com/old/cookbook/fine-tuning-examples

https://www.youtube.com/watch?v=h0OPWlEOank&list=PLfaIDFEXuae2LXbO1_PKyVJiQ23ZztA0x&index=7s

A typical RAG application has two main components:

Indexing: a pipeline for ingesting data from a source and indexing it. This usually happens offline.

Retrieval and generation: the actual RAG chain, which takes the user query at run time and retrieves the relevant data from the index, then passes that to the model.

a .Indexing
1. load
2. split
3. embeded and store

b. Retrieval and generation
1. Retrieve
2. generate the prompt

c. Process the document
d. Create a chain



In [2]:
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
model=ChatOpenAI(api_key=os.getenv('token'),model=os.getenv('model_name'))
embd=OpenAIEmbeddings(api_key=os.getenv('token'),model='text-embedding-3-small')

In [4]:
#1. Indexing-load document
pdf=PyPDFLoader(file_path='../data/guidlines.pdf')
pdfSplit=pdf.load_and_split()
pdfSplit


[Document(metadata={'source': '../data/guidlines.pdf', 'page': 0}, page_content="Scala Coding Guidelines  \nQuantexa Coding Guidelines:  \nGeneral \nScala/Spark   \nCamel Case  Use camel casing, not underscores! Vals/vars/methods(defs) should all \nbe lowerCamelCase, types/objects should all be UpperCamelCase (also \nknown as ProperCase), for further details please  see this style guide .  \nExceptions  \n1. In ETL processes, the raw data should be read into a parquet \nmaintaining all of the source field names exactly.   \n2. Constant values - use ProperCase. For example, val YearInDays = \n365 \n3. When using acronyms and camel case, if it is the first word \nwithin a variable name it should be lower case, if it is not the first \nword, it should be all upper case. E.G:  \no jiraTaskDescription (and not JIRATaskDescription)  \no accountIBAN (and not accountIban)  \n4. When using acronyms and proper case,  every letter of the \nacronym should be upper case, e.g.  \no case class ETLCon

In [5]:
#2. Indexing-Split the loading document
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits=splitter.split_documents(pdfSplit)
splits


[Document(metadata={'source': '../data/guidlines.pdf', 'page': 0}, page_content='Scala Coding Guidelines  \nQuantexa Coding Guidelines:  \nGeneral \nScala/Spark   \nCamel Case  Use camel casing, not underscores! Vals/vars/methods(defs) should all \nbe lowerCamelCase, types/objects should all be UpperCamelCase (also \nknown as ProperCase), for further details please  see this style guide .  \nExceptions  \n1. In ETL processes, the raw data should be read into a parquet \nmaintaining all of the source field names exactly.   \n2. Constant values - use ProperCase. For example, val YearInDays = \n365 \n3. When using acronyms and camel case, if it is the first word \nwithin a variable name it should be lower case, if it is not the first \nword, it should be all upper case. E.G:  \no jiraTaskDescription (and not JIRATaskDescription)  \no accountIBAN (and not accountIban)  \n4. When using acronyms and proper case,  every letter of the \nacronym should be upper case, e.g.  \no case class ETLCon

In [6]:
#3. Indexing-Store the splits into vectorstores
vectorStore=Chroma.from_documents(embedding=embd,documents=splits)
vectorStore

In [7]:
# retriver and generation -Retriever
retriever=vectorStore.as_retriever()
retriever


VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000027C51146390>)

In [8]:
# retriver and generation -GENERATE
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
  (
"human", """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
),
])
prompt

ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:\n"))])

In [9]:
example_messages = prompt.invoke(
    {"context": "filler context", "question": "filler question"}
).to_messages()

print(example_messages[0].content)


You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
Question: filler question 
Context: filler context 
Answer:



In [10]:
#c. Process the document
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [11]:
format_docs(splits)

'Scala Coding Guidelines  \nQuantexa Coding Guidelines:  \nGeneral \nScala/Spark   \nCamel Case  Use camel casing, not underscores! Vals/vars/methods(defs) should all \nbe lowerCamelCase, types/objects should all be UpperCamelCase (also \nknown as ProperCase), for further details please  see this style guide .  \nExceptions  \n1. In ETL processes, the raw data should be read into a parquet \nmaintaining all of the source field names exactly.   \n2. Constant values - use ProperCase. For example, val YearInDays = \n365 \n3. When using acronyms and camel case, if it is the first word \nwithin a variable name it should be lower case, if it is not the first \nword, it should be all upper case. E.G:  \no jiraTaskDescription (and not JIRATaskDescription)  \no accountIBAN (and not accountIban)  \n4. When using acronyms and proper case,  every letter of the \nacronym should be upper case, e.g.  \no case class ETLConfig(hdfsPath: String)  \n5. Packages all lower case, including package objects. 

In [12]:
#d. ragchain- retriever-->prompt-->model-->output
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

for chunk in rag_chain.stream("Scala  Coding Guidelines",):
    print(chunk, end="", flush=True)

Scala coding guidelines recommend the use of camel casing, lowerCamelCase for Vals/vars/methods, and UpperCamelCase for types/objects. Acronyms should follow specific rules based on their position within a variable name. Comments should be minimized by writing clear and functional code that reads like a story.

In [14]:
splits

[Document(metadata={'source': '../data/guidlines.pdf', 'page': 0}, page_content='Scala Coding Guidelines  \nQuantexa Coding Guidelines:  \nGeneral \nScala/Spark   \nCamel Case  Use camel casing, not underscores! Vals/vars/methods(defs) should all \nbe lowerCamelCase, types/objects should all be UpperCamelCase (also \nknown as ProperCase), for further details please  see this style guide .  \nExceptions  \n1. In ETL processes, the raw data should be read into a parquet \nmaintaining all of the source field names exactly.   \n2. Constant values - use ProperCase. For example, val YearInDays = \n365 \n3. When using acronyms and camel case, if it is the first word \nwithin a variable name it should be lower case, if it is not the first \nword, it should be all upper case. E.G:  \no jiraTaskDescription (and not JIRATaskDescription)  \no accountIBAN (and not accountIban)  \n4. When using acronyms and proper case,  every letter of the \nacronym should be upper case, e.g.  \no case class ETLCon

In [15]:
seq=retriever|format_docs
seq.invoke("Scala  Coding Guidelines")

'Scala Coding Guidelines  \nQuantexa Coding Guidelines:  \nGeneral \nScala/Spark   \nCamel Case  Use camel casing, not underscores! Vals/vars/methods(defs) should all \nbe lowerCamelCase, types/objects should all be UpperCamelCase (also \nknown as ProperCase), for further details please  see this style guide .  \nExceptions  \n1. In ETL processes, the raw data should be read into a parquet \nmaintaining all of the source field names exactly.   \n2. Constant values - use ProperCase. For example, val YearInDays = \n365 \n3. When using acronyms and camel case, if it is the first word \nwithin a variable name it should be lower case, if it is not the first \nword, it should be all upper case. E.G:  \no jiraTaskDescription (and not JIRATaskDescription)  \no accountIBAN (and not accountIban)  \n4. When using acronyms and proper case,  every letter of the \nacronym should be upper case, e.g.  \no case class ETLConfig(hdfsPath: String)  \n5. Packages all lower case, including package objects. 

In [16]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    
)
chain.stream("What is the syntax reccomendation")

AttributeError: 'dict' object has no attribute 'stream'

In [ ]:
rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000026558BA7B90>)
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:\n"))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x00000265733B5250>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x00000265733B6FC0>, root_client=<openai.OpenAI object at 0x0000026573373050>, root_async_client=<openai.AsyncOpenAI obj